# NEHM Student inference (Pipeline **B v3.1**, LongCLIP-B 16+1 *scale-adaptive* view) → UMAP / HDBSCAN / FAISS / Bayesian + K-SVD

**Pipeline B v3.1** — student trained on **LongCLIP-B (ViT-B/16, 512-D)** with the same 17-view
layout as the B v3.1 teacher (`output_random_local_global_B/`): **1 global 224×224 view + 16
random scale-adaptive native-resolution tiles** whose per-image side is

```
tile_px = clip(round(TILE_MAX_PX × sqrt(W*H) / TILE_SCALE_REF_PX), TILE_MIN_PX, TILE_MAX_PX)
```

(defaults `TILE_MIN_PX = 16`, `TILE_MAX_PX = 224`, `TILE_SCALE_REF_PX = 3000`). The largest
images (~3000-px geometric-mean side, e.g. McCrone) get full **224×224** tiles; the smallest
images (~150-px, e.g. Emrath) saturate at **16×16** tiles; everything in between is sized so
each tile samples a roughly constant fraction of the image's *physical FOV*.

The global view drops a 1-pixel scan border → short-side resize to 224 → center-crop 224×224. RGB
stays linear in `[0, 1]` with **no per-channel normalization** (the framework default — preserves
microscopy-relevant per-channel intensity ratios). This notebook runs that exact preprocessing at
inference time, then writes B-suffixed npys so A and B inference outputs coexist. Set `TILE_PX > 0`
in the config cell to fall back to the legacy fixed-size B v3 layout.

**What this notebook does**

1. Loads `vit_student_finetune_best_B_logitb.pth` (the global-best B v3 student). Falls back
   through B-stage checkpoints if the global-best is missing.
2. Runs **17-view inference** under LongCLIP-B preprocessing — exactly the training-time
   geometry: 16 random tiles (mean-pooled visual embedding) + 1 global view (also pooled in via
   `batched_pooled_visual_embeddings`).
3. Saves three arrays per row, aligned with `Material_Database.xlsx` row order:
    - `student_image_emb_768_B.npy` (N, 512)  — LongCLIP-B visual embedding (multi-view pool, L2-normalized).
    - `student_z_2304_B.npy`        (N, 2048) — student decoder reconstruction of the fused **B v3** teacher target.
    - `student_logits_B.npy`        (N, K)    — classifier logits for `logitb`.
4. Builds `Z = student_z_2304_B` and runs UMAP → HDBSCAN with the existing visualization.
5. FAISS nearest-neighbor panel + Bayesian inference table + generative-style summary.
6. **K-SVD dictionary learning** on the 2304-D student z (mirrors
   `Painting_Interpretability_Study/Stable/Notebooks/Student_Embed_KSVD.ipynb`):
   sparse dictionary on the L2-normalized student embeddings, reconstruction overlay,
   atom shape plots, and per-atom decoded classifier-class footprints.

**Runtime.** 17 LongCLIP-B forwards per row on the smaller ViT-B/16 backbone is roughly the
same wall clock as Pipeline A (~minutes on A100, ~30-60 min on MPS for the full ~5,400 rows).
The downstream visualization cells are unchanged from A.

In [ ]:
from __future__ import annotations
import os
import sys
import importlib
import importlib.util
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def _detect_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "nehm_pipeline").is_dir() and (cand / "Database").is_dir():
            return cand
    drive_root = Path("/content/drive/MyDrive/NEHM_Project")
    if drive_root.is_dir():
        return drive_root
    raise FileNotFoundError("Could not auto-detect PROJECT_ROOT. Set it manually.")

PROJECT_ROOT = _detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MATERIAL_DB_XLSX = PROJECT_ROOT / "Database" / "Material_Database.xlsx"
IMAGES_DIR = PROJECT_ROOT / "NEHM"
# STUDENT_CKPT (Pipeline B): first-match-wins among the B-suffixed checkpoint names below.
# All names match those written by Colab/Student_Trainer_B.ipynb (B v2).
_STUDENT_CKPT_CANDIDATES = [
    "vit_student_finetune_best_B_logitb.pth",
    "vit_student_finetune_best_B.pth",
    "vit_finetune_stage_B_D_full_decay.pth",
    "vit_finetune_stage_B_C_last12.pth",
    "vit_finetune_stage_B_C_last8.pth",
    "vit_finetune_stage_B_C_last4.pth",
    "vit_finetune_stage_B_B_last2.pth",
    "vit_finetune_stage_B_A_readout.pth",
    "vit_finetune_stage_B_Z_embed_student_head.pth",
]
STUDENT_CKPT = next(
    (PROJECT_ROOT / "NEHM_RESULTS" / _n for _n in _STUDENT_CKPT_CANDIDATES
     if (PROJECT_ROOT / "NEHM_RESULTS" / _n).is_file()),
    PROJECT_ROOT / "NEHM_RESULTS" / _STUDENT_CKPT_CANDIDATES[0],
)

CLIP_MODEL = "LongCLIP-B"        # metadata label for prints / CSVs
EXPECTED_IMAGE_DIM = 768         # LongCLIP-B ViT-B/16: pooled ln_post CLS (no visual.proj)
EXPECTED_TEXT_BLOCK_DIM = 512    # text EOT after ln_final (no text_projection)
EXPECTED_TEACHER_DIM = EXPECTED_IMAGE_DIM + 3 * EXPECTED_TEXT_BLOCK_DIM  # 2304

# LongCLIP package (vendored as a relative-import package — the loader in cell 3 wires sys.path).
LONGCLIP_ROOT = PROJECT_ROOT / "LongCLIP"
LONGCLIP_CHECKPOINT = LONGCLIP_ROOT / "checkpoints" / "longclip-B.pt"

EMB_OUT_DIR = PROJECT_ROOT / "NEHM_RESULTS" / "student_inference_B"
EMB_OUT_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_IMG_EMB_NPY = EMB_OUT_DIR / "student_image_emb_768_B.npy"
STUDENT_Z_NPY       = EMB_OUT_DIR / "student_z_2304_B.npy"
STUDENT_LOGITS_NPY  = EMB_OUT_DIR / "student_logits_B.npy"
STUDENT_PRED_CSV    = EMB_OUT_DIR / "student_predictions_B.csv"

# Pipeline B v3 view layout (must match Colab/Student_Trainer_B.ipynb + Teacher notebook).
MATERIAL_LOGIT_COLUMN = "logitb"
N_NATIVE_RANDOM     = 16          # B v3.1: 16 random native tiles per image (matches Pipeline A)
TILE_PX             = 0           # 0 = scale-adaptive (B v3.1); >0 = legacy fixed-size (B v3)
TILE_MIN_PX         = 16          # smallest adaptive tile side (matches ViT-B/16 patch grid)
TILE_MAX_PX         = 224         # largest adaptive tile side (matches LongCLIP-B input)
TILE_SCALE_REF_PX   = 3000        # sqrt(W*H) image side that maps to TILE_MAX_PX
GLOBAL_KIND         = "short_side"
WINDOW_SAMPLE_SEED  = 42
CROP_SIZE           = 224         # LongCLIP-B native input (used for the global view + ViT preprocess)
PRE_CROP_MARGIN_PX  = 1           # B v3: drop 1px scan border before resize (matches teacher + trainer)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

# B v3 = 17 LongCLIP-B forwards per row (16 random tiles + 1 global). Smaller batches than
# the global-only B v2 default because the per-sample tile fan-out reappears, but the smaller
# ViT-B/16 backbone (vs ViT-L/14@336) still lets us push 16-32 on A100 / 8 on MPS.
if DEVICE.type == "cuda":
    INFER_BATCH_SIZE = 16
    VISUAL_ENCODE_CHUNK_SIZE = 32
    DATALOADER_NUM_WORKERS = 4
elif DEVICE.type == "mps":
    INFER_BATCH_SIZE = 4
    VISUAL_ENCODE_CHUNK_SIZE = 16
    DATALOADER_NUM_WORKERS = 0
else:
    INFER_BATCH_SIZE = 2
    VISUAL_ENCODE_CHUNK_SIZE = 8
    DATALOADER_NUM_WORKERS = 0

print(f"PROJECT_ROOT       : {PROJECT_ROOT}")
print(f"MATERIAL_DB_XLSX   : {MATERIAL_DB_XLSX}")
print(f"IMAGES_DIR         : {IMAGES_DIR}")
print(f"STUDENT_CKPT       : {STUDENT_CKPT}  exists={STUDENT_CKPT.is_file()}")
print(f"CLIP_MODEL         : {CLIP_MODEL}  (LongCLIP-B; checkpoint @ {LONGCLIP_CHECKPOINT})")
print(f"EMB_OUT_DIR        : {EMB_OUT_DIR}")
print(f"DEVICE             : {DEVICE}  | infer_batch={INFER_BATCH_SIZE}  encode_chunk={VISUAL_ENCODE_CHUNK_SIZE}  workers={DATALOADER_NUM_WORKERS}")

for required in (MATERIAL_DB_XLSX, IMAGES_DIR, STUDENT_CKPT, LONGCLIP_CHECKPOINT):
    if not Path(required).exists():
        raise FileNotFoundError(f"Missing required input: {required}")

In [ ]:
_df_raw = pd.read_excel(MATERIAL_DB_XLSX, sheet_name="Sheet2")
if "Image Index" not in _df_raw.columns:
    raise ValueError(f"Expected 'Image Index' in {MATERIAL_DB_XLSX} sheet 'Sheet2'")

_df_manifest = _df_raw.reset_index(drop=True).copy()
N = int(len(_df_manifest))
row_indices = np.arange(N, dtype=np.int64)

MANIFEST_KEYS = _df_manifest["Image Index"].astype(np.int64).to_numpy()
if "Image" in _df_manifest.columns:
    filenames = _df_manifest["Image"].fillna("").astype(str).to_numpy()
else:
    filenames = np.array(["" for _ in range(N)], dtype=object)

y_sheet_default = np.full(N, -1, dtype=np.int64)
if MATERIAL_LOGIT_COLUMN in _df_manifest.columns:
    _y_raw = _df_manifest[MATERIAL_LOGIT_COLUMN].astype("Int64").fillna(-1).to_numpy(dtype=np.int64)
    y_sheet = _y_raw
else:
    y_sheet = y_sheet_default.copy()

print(f"Rows loaded: {N}")
print(f"Image filenames present: {int((np.char.str_len(filenames.astype(str)) > 0).sum())} / {N}")
print(f"{MATERIAL_LOGIT_COLUMN!r} labeled rows (>=0): {int((y_sheet >= 0).sum())}")

In [ ]:
import nehm_pipeline.preprocess_image as preimg
import nehm_pipeline.clip_vit_multiview_finetune as vfin
import nehm_pipeline.clip_visual_student as cvs
import nehm_pipeline.student_clip as sc

for _m in (preimg, vfin, cvs, sc):
    importlib.reload(_m)

from nehm_pipeline.clip_visual_student import (
    ClipVisualStudent,
    load_clip_visual_student_checkpoint,
)
from nehm_pipeline.student_clip import ImageEmbeddingStudent
from nehm_pipeline.device import clip_visual_preprocess_without_channel_normalize


# Robust LongCLIP package loader (mirrors the teacher / trainer notebooks). The package uses
# relative imports (`from .model_longclip import build_model`), so the model/ directory must
# be loaded as a package, not as a loose file.
def _load_longclip_module():
    pkg_dir = LONGCLIP_ROOT / "model"
    init_py = pkg_dir / "__init__.py"
    longclip_py = pkg_dir / "longclip.py"
    model_longclip_py = pkg_dir / "model_longclip.py"
    simple_tok_py = pkg_dir / "simple_tokenizer.py"
    missing = [p for p in (pkg_dir, init_py, longclip_py, model_longclip_py, simple_tok_py) if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "LongCLIP package incomplete; missing:\n  " + "\n  ".join(str(m) for m in missing)
        )
    if str(LONGCLIP_ROOT) not in sys.path:
        sys.path.insert(0, str(LONGCLIP_ROOT))
    for k in [k for k in list(sys.modules) if k == "model" or k.startswith("model.")]:
        del sys.modules[k]
    try:
        from model import longclip as _lc  # type: ignore
        return _lc
    except Exception as _err:
        print(f"  (simple `from model import longclip` failed: {_err}; falling back to importlib)")
    pkg_spec = importlib.util.spec_from_file_location(
        "model", str(init_py), submodule_search_locations=[str(pkg_dir)],
    )
    pkg_mod = importlib.util.module_from_spec(pkg_spec)
    sys.modules["model"] = pkg_mod
    pkg_spec.loader.exec_module(pkg_mod)
    sub_spec = importlib.util.spec_from_file_location("model.longclip", str(longclip_py))
    sub_mod = importlib.util.module_from_spec(sub_spec)
    sys.modules["model.longclip"] = sub_mod
    sub_spec.loader.exec_module(sub_mod)
    return sub_mod


longclip_module = _load_longclip_module()
print(f"LongCLIP loaded from: {LONGCLIP_ROOT / 'model' / 'longclip.py'}")

# LongCLIP-B (ViT-B/16, 768-D ln_post). channel_normalize=False keeps RGB linear in [0, 1]
# (NEHM framework default — preserves microscopy-relevant per-channel intensity ratios). Must
# match the trainer's setting; flipping it without retraining will silently degrade embeddings.
_clip_model, _preprocess_with_normalize = longclip_module.load(str(LONGCLIP_CHECKPOINT), device=str(DEVICE))
_clip_model = _clip_model.float().to(DEVICE)
preprocess = clip_visual_preprocess_without_channel_normalize(_preprocess_with_normalize)
print("LongCLIP-B preprocess: Normalize stripped (linear RGB in [0, 1])")

visual = _clip_model.visual
visual.eval()

from nehm_pipeline.longclip_image_embeddings import vit_visual_ln_post_forward
with torch.inference_mode():
    _res = int(getattr(visual, "input_resolution", CROP_SIZE))
    _probe_in = torch.zeros(1, 3, _res, _res, device=DEVICE)
    _img_dim = int(vit_visual_ln_post_forward(visual, _probe_in).shape[-1])
print(f"LongCLIP-B visual probe ({CLIP_MODEL}): ln_post output_dim={_img_dim} (expected {EXPECTED_IMAGE_DIM})")
if _img_dim != EXPECTED_IMAGE_DIM:
    raise ValueError(
        f"Visual ln_post dim {_img_dim} != EXPECTED_IMAGE_DIM {EXPECTED_IMAGE_DIM}. "
        f"Checkpoint must be ViT-B/16 width 768 for Pipeline B v3.1 (pre-projection teacher)."
    )

_ckpt_meta = torch.load(STUDENT_CKPT, map_location="cpu")
_meta = dict(_ckpt_meta.get("meta", {}))
_n_classes_ckpt = int(_meta.get("n_material_classes") or 0)
_teacher_dim_ckpt = int(_meta.get("teacher_embed_dim") or 0)
if _teacher_dim_ckpt <= 0:
    sd = _ckpt_meta.get("state_dict") or {}
    for k, v in sd.items():
        if k.endswith("student.classifier.weight"):
            _n_classes_ckpt = int(v.shape[0])
            _teacher_dim_ckpt = int(v.shape[1])
            break
if _n_classes_ckpt <= 0 or _teacher_dim_ckpt <= 0:
    raise ValueError(f"Could not infer student dims from checkpoint meta or state_dict: {STUDENT_CKPT}")
if _teacher_dim_ckpt != EXPECTED_TEACHER_DIM:
    raise ValueError(
        f"Checkpoint teacher_embed_dim={_teacher_dim_ckpt} != EXPECTED_TEACHER_DIM={EXPECTED_TEACHER_DIM}. "
        f"This .pth was trained against a different fused-teacher size; "
        f"use a B v3 checkpoint ({EXPECTED_TEACHER_DIM}-D fused teacher) or rerun training."
    )
print(f"Checkpoint meta : n_material_classes={_n_classes_ckpt}  teacher_embed_dim={_teacher_dim_ckpt}")

# B v3 student head shape: 512 -> 1024 -> 1536 -> 2304 (decoder), then Linear(2304 -> K).
student = ImageEmbeddingStudent(
    image_dim=_img_dim,
    num_classes=_n_classes_ckpt,
    embed_dim=_teacher_dim_ckpt,
    decoder_hidden=(1024, 1536),
    dropout=0.0,
    head_style="decoder",
).to(DEVICE).eval()

full_model = ClipVisualStudent(visual, student).to(DEVICE).eval()
load_clip_visual_student_checkpoint(STUDENT_CKPT, full_model, map_location=DEVICE, strict=True)
for p in full_model.parameters():
    p.requires_grad = False
print(f"Loaded student checkpoint OK → image_dim={_img_dim}  z_dim={_teacher_dim_ckpt}  classes={_n_classes_ckpt}")

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from nehm_pipeline.clip_vit_multiview_finetune import (
    LabeledMultiviewFinetuneDataset,
    batched_pooled_visual_ln_post_embeddings,
    collate_multiview_pil_teacher,
)

import inspect as _inspect_b
_lmd_params_b = _inspect_b.signature(LabeledMultiviewFinetuneDataset.__init__).parameters
for _need in ("tile_px", "global_kind", "pre_crop_margin_px"):
    if _need not in _lmd_params_b:
        raise RuntimeError(
            f"LabeledMultiviewFinetuneDataset on this Python path lacks `{_need}`. "
            "Update nehm_pipeline/clip_vit_multiview_finetune.py from NEHM_Stable — "
            "Pipeline B inference requires it (see README_B.md)."
        )
for _need in ("tile_min_px", "tile_max_px", "tile_scale_ref_px"):
    if _need not in _lmd_params_b:
        raise RuntimeError(
            f"LabeledMultiviewFinetuneDataset on this Python path lacks `{_need}`. "
            "Update nehm_pipeline/clip_vit_multiview_finetune.py from NEHM_Stable — "
            "Pipeline B v3.1 (scale-adaptive multiview) requires it (see README_B.md)."
        )

_dummy_teacher = np.zeros((N, _teacher_dim_ckpt), dtype=np.float32)
ds_all = LabeledMultiviewFinetuneDataset(
    row_indices,
    y_sheet,
    filenames,
    IMAGES_DIR,
    n_native_random=N_NATIVE_RANDOM,
    crop_size=CROP_SIZE,
    fused_teacher=_dummy_teacher,
    window_sample_seed=WINDOW_SAMPLE_SEED,
    geom_augment=False,
    tile_px=TILE_PX,
    global_kind=GLOBAL_KIND,
    pre_crop_margin_px=PRE_CROP_MARGIN_PX,
    tile_min_px=TILE_MIN_PX,
    tile_max_px=TILE_MAX_PX,
    tile_scale_ref_px=TILE_SCALE_REF_PX,
)
if N_NATIVE_RANDOM == 0:
    print(
        f"B inference view layout (legacy global-only): 1 global view only "
        f"({GLOBAL_KIND}, {CROP_SIZE}x{CROP_SIZE}, pre_crop_margin_px={PRE_CROP_MARGIN_PX})."
    )
elif int(TILE_PX) > 0:
    print(
        f"B v3 inference view layout: 1 global ({GLOBAL_KIND}, {CROP_SIZE}x{CROP_SIZE}, "
        f"pre_crop_margin_px={PRE_CROP_MARGIN_PX}) + "
        f"{N_NATIVE_RANDOM} random {TILE_PX}x{TILE_PX} fixed-size native tiles per row "
        f"= {1 + N_NATIVE_RANDOM} views/sample (must match Colab/Student_Trainer_B.ipynb)."
    )
else:
    print(
        f"B v3.1 inference view layout: 1 global ({GLOBAL_KIND}, {CROP_SIZE}x{CROP_SIZE}, "
        f"pre_crop_margin_px={PRE_CROP_MARGIN_PX}) + "
        f"{N_NATIVE_RANDOM} random scale-adaptive native tiles "
        f"[{TILE_MIN_PX}-{TILE_MAX_PX}px @ ref={TILE_SCALE_REF_PX}] per row "
        f"= {1 + N_NATIVE_RANDOM} views/sample (must match Colab/Student_Trainer_B.ipynb)."
    )

_loader_kwargs = dict(
    batch_size=INFER_BATCH_SIZE,
    shuffle=False,
    num_workers=int(DATALOADER_NUM_WORKERS),
    collate_fn=collate_multiview_pil_teacher,
    pin_memory=(DEVICE.type == "cuda"),
)
if DATALOADER_NUM_WORKERS > 0:
    _loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
ld_all = DataLoader(ds_all, **_loader_kwargs)

from contextlib import nullcontext

_use_cuda_amp = DEVICE.type == "cuda"
_amp_ctx = torch.amp.autocast("cuda", dtype=torch.float16) if _use_cuda_amp else nullcontext()

image_emb_all = np.empty((N, _img_dim), dtype=np.float32)
z_all = np.empty((N, _teacher_dim_ckpt), dtype=np.float32)
logits_all = np.empty((N, _n_classes_ckpt), dtype=np.float32)

row_cursor = 0
with torch.inference_mode():
    for crops_b, _t, _y in tqdm(ld_all, desc="student inference", total=len(ld_all)):
        bs = len(crops_b)
        with _amp_ctx:
            emb = batched_pooled_visual_ln_post_embeddings(
                visual,
                preprocess,
                crops_b,
                DEVICE,
                encode_chunk_size=VISUAL_ENCODE_CHUNK_SIZE,
            )
            z, logits = student(emb)
        image_emb_all[row_cursor : row_cursor + bs] = emb.detach().to("cpu", dtype=torch.float32).numpy()
        z_all[row_cursor : row_cursor + bs] = z.detach().to("cpu", dtype=torch.float32).numpy()
        logits_all[row_cursor : row_cursor + bs] = logits.detach().to("cpu", dtype=torch.float32).numpy()
        row_cursor += bs

if row_cursor != N:
    raise RuntimeError(f"Inference produced {row_cursor} rows, expected {N}")

np.save(STUDENT_IMG_EMB_NPY, image_emb_all)
np.save(STUDENT_Z_NPY, z_all)
np.save(STUDENT_LOGITS_NPY, logits_all)

_pred_top1 = np.argmax(logits_all, axis=1).astype(np.int64)
_pred_prob = (np.exp(logits_all - logits_all.max(axis=1, keepdims=True)).astype(np.float64))
_pred_prob = _pred_prob / np.maximum(_pred_prob.sum(axis=1, keepdims=True), 1e-12)
_pred_top1_prob = _pred_prob[np.arange(N), _pred_top1]

pred_df = pd.DataFrame({
    "row": np.arange(N, dtype=np.int64),
    "image_index": MANIFEST_KEYS,
    "image": filenames,
    "y_true_logitb": y_sheet,
    "y_pred_logitb": _pred_top1,
    "y_pred_prob": _pred_top1_prob.astype(np.float32),
    "correct": np.where(y_sheet >= 0, (y_sheet == _pred_top1).astype(np.int8), -1),
})
pred_df.to_csv(STUDENT_PRED_CSV, index=False)

print(f"image_emb_{_img_dim}: {image_emb_all.shape}  -> {STUDENT_IMG_EMB_NPY}")
print(f"z_{_teacher_dim_ckpt}       : {z_all.shape}  -> {STUDENT_Z_NPY}")
print(f"logits       : {logits_all.shape}  -> {STUDENT_LOGITS_NPY}")
print(f"predictions  : {STUDENT_PRED_CSV}")
lab = y_sheet >= 0
if lab.any():
    acc_all = float((y_sheet[lab] == _pred_top1[lab]).mean())
    print(f"top-1 acc on labeled rows ({int(lab.sum())}/{N}): {acc_all:.4f}")

In [ ]:
# CLEAN UMAP + HDBSCAN on student z (2304-D, B v3 LongCLIP-B). Same plot/visualization as the base notebook.
import umap
import hdbscan
import plotly.graph_objects as go
from sklearn.preprocessing import normalize

Z = np.load(STUDENT_Z_NPY).astype(np.float32) if not ("z_all" in globals()) else z_all
if Z.shape[0] != N:
    raise ValueError(f"Z rows {Z.shape[0]} != manifest N {N}")

UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0
UMAP_METRIC = "cosine"
CLUSTER_MIN_SIZE = 12
CLUSTER_MIN_SAMPLES = 4

Zn = normalize(Z, norm="l2", axis=1)

umap5 = umap.UMAP(
    n_components=5,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=42,
)
z5 = umap5.fit_transform(Zn)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=CLUSTER_MIN_SIZE,
    min_samples=CLUSTER_MIN_SAMPLES,
    metric="euclidean",
    cluster_selection_method="leaf",
)
hdb_labels = clusterer.fit_predict(z5)

umap2 = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=42,
)
xy = umap2.fit_transform(Zn)

def _safe_col(col):
    if col not in _df_manifest.columns:
        return np.array(["" for _ in range(N)], dtype=object)
    return _df_manifest[col].fillna("").astype(str).to_numpy(dtype=object)

spec = _safe_col("Specimen Name")
illum_col = _safe_col("Illumination Modality")
subclass = _safe_col("Subclass")
desc = _safe_col("Description")
cluster_txt = np.array([str(v) for v in hdb_labels], dtype=object)
img_idx_txt = np.array([str(v) for v in MANIFEST_KEYS], dtype=object)
custom = np.column_stack([spec, illum_col, subclass, desc, cluster_txt])

fig = go.Figure()
noise = hdb_labels == -1
if noise.any():
    fig.add_trace(
        go.Scattergl(
            x=xy[noise, 0], y=xy[noise, 1], mode="markers", name="noise (-1)",
            marker=dict(size=4, color="#b0b0b0", opacity=0.65),
            customdata=custom[noise], text=img_idx_txt[noise],
            hovertemplate=(
                "<b>Image Index</b>: %{text}<br>"
                "<b>Specimen</b>: %{customdata[0]}<br>"
                "<b>Illumination</b>: %{customdata[1]}<br>"
                "<b>Subclass</b>: %{customdata[2]}<br>"
                "<b>Description</b>: %{customdata[3]}<br>"
                "<b>HDBSCAN</b>: %{customdata[4]}<extra></extra>"
            ),
        )
    )
for cid in sorted(np.unique(hdb_labels[hdb_labels >= 0])):
    m = hdb_labels == cid
    fig.add_trace(
        go.Scattergl(
            x=xy[m, 0], y=xy[m, 1], mode="markers", name=f"cluster {cid}",
            marker=dict(size=4, opacity=0.85),
            customdata=custom[m], text=img_idx_txt[m],
            hovertemplate=(
                "<b>Image Index</b>: %{text}<br>"
                "<b>Specimen</b>: %{customdata[0]}<br>"
                "<b>Illumination</b>: %{customdata[1]}<br>"
                "<b>Subclass</b>: %{customdata[2]}<br>"
                "<b>Description</b>: %{customdata[3]}<br>"
                "<b>HDBSCAN</b>: %{customdata[4]}<extra></extra>"
            ),
        )
    )
fig.update_layout(
    title=f"Student z ({Z.shape[1]}-D, B v3 LongCLIP-B) → UMAP (2D) colored by HDBSCAN",
    xaxis_title="UMAP-1", yaxis_title="UMAP-2",
    height=850, legend_title="HDBSCAN",
)
fig.show()

n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
noise_frac = float((hdb_labels == -1).sum()) / float(N)
print(f"Clusters: {n_clusters} | noise fraction: {noise_frac:.3f} | points: {N}")

In [ ]:
# Persist HDBSCAN cluster assignments and 2D UMAP coordinates per row.
HDBSCAN_CSV_OUT = EMB_OUT_DIR / "hdbscan_clusters_student.csv"
save_df = _df_manifest.copy()
save_df["hdbscan_cluster"] = hdb_labels.astype(np.int64)
save_df["umap2_x"] = xy[:, 0].astype(np.float32)
save_df["umap2_y"] = xy[:, 1].astype(np.float32)
save_df.to_csv(HDBSCAN_CSV_OUT, index=False)
print(f"saved: {HDBSCAN_CSV_OUT}")

In [ ]:
# THUMBNAIL UMAP OVERLAY (sample of points placed as image thumbnails).
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from PIL import Image

MAX_THUMBS = 860
ZOOM = 0.10

rng = np.random.default_rng(42)
perm = rng.permutation(N)
idx = perm[:MAX_THUMBS]
xs = xy[idx, 0]
ys = xy[idx, 1]

fig, ax = plt.subplots(figsize=(11, 9))
ax.set_title("UMAP-2D thumbnail overlay (student z)")
ax.set_xlim(xy[:, 0].min() - 0.5, xy[:, 0].max() + 0.5)
ax.set_ylim(xy[:, 1].min() - 0.5, xy[:, 1].max() + 0.5)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.scatter(xy[:, 0], xy[:, 1], s=2, alpha=0.25, color="#a0a0a0")

def _thumb_rgb(p: Path) -> Image.Image:
    im = Image.open(p).convert("RGB")
    im.thumbnail((96, 96))
    return im

placed = 0
missing = 0
for i, x0, y0 in zip(idx, xs, ys):
    img_name = str(filenames[i]).strip() if i < len(filenames) else ""
    if not img_name:
        missing += 1
        continue
    p = IMAGES_DIR / img_name
    if not p.is_file():
        missing += 1
        continue
    try:
        ab = AnnotationBbox(OffsetImage(_thumb_rgb(p), zoom=ZOOM, alpha=1.0), (x0, y0), frameon=False)
        ax.add_artist(ab)
        placed += 1
    except Exception:
        missing += 1

plt.tight_layout()
plt.show()
print(f"Placed thumbnails: {placed} | missing/failed: {missing}")

In [ ]:
# CLEAN FAISS NEIGHBOR PANEL (student z). Shows query + nearest neighbors.
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

try:
    import faiss  # type: ignore
except ModuleNotFoundError:
    print("Installing faiss-cpu ...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    import faiss  # type: ignore

QUERY_ROW = 1742

N_NEIGH = 9
THUMB_SIZE = 213

if Z.shape[0] != N:
    raise ValueError(f"Z rows {Z.shape[0]} != N {N}")

xb = np.ascontiguousarray(np.asarray(Z, dtype=np.float32))
index = faiss.IndexFlatL2(xb.shape[1])
index.add(xb)

q = int(QUERY_ROW) % int(N)
search_k = min(N_NEIGH + 2, int(N))
dist2, idx = index.search(xb[q:q+1], search_k)

hits = []
for ix, d2 in zip(idx[0].tolist(), dist2[0].tolist()):
    if ix < 0 or ix == q:
        continue
    hits.append((int(ix), float(d2)))
    if len(hits) >= N_NEIGH:
        break

rows = [q] + [h[0] for h in hits]

def _safe_cell(i: int, col: str) -> str:
    if col not in _df_manifest.columns:
        return ""
    v = _df_manifest.iloc[int(i)][col]
    if isinstance(v, float) and np.isnan(v):
        return ""
    return str(v).strip()

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.ravel()

for j, ridx in enumerate(rows[:10]):
    ax = axes[j]
    img_name = str(filenames[ridx]) if ridx < len(filenames) else ""
    p = (IMAGES_DIR / img_name) if img_name else None

    if p and Path(p).is_file():
        try:
            im = Image.open(p).convert("RGB")
            im.thumbnail((THUMB_SIZE, THUMB_SIZE))
            ax.imshow(im)
        except Exception:
            ax.text(0.5, 0.5, "image load error", ha="center", va="center")
    else:
        ax.text(0.5, 0.5, "missing image", ha="center", va="center")

    desc = _safe_cell(ridx, "Description")
    illum = _safe_cell(ridx, "Illumination Modality")
    desc_head = (desc[:52] + "…") if len(desc) > 52 else desc
    if not desc_head:
        desc_head = "(no description)"

    if j == 0:
        ax.set_title(f"{desc_head}\nquery row {ridx}", fontsize=9)
    else:
        ax.set_title(f"{desc_head}\nrow {ridx} | d2={hits[j-1][1]:.3f}", fontsize=9)

    ax.set_xlabel(f"illum: {illum[:48]}", fontsize=8)
    ax.axis("off")

for j in range(len(rows), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()
print(f"FAISS query row={q}, neighbors shown={len(rows)-1}")

In [ ]:
# BAYESIAN TABLE (matplotlib layout: image left, table right). Uses FAISS index built above.
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

K_BAYES = 200
alpha0 = 0.85
TAU_FRAC = 0.5
NEIGHBOR_TEMP = 1.0
TOP1_BOOST = 1.0

FIELD_COLUMNS = [
    "Class",
    "Subclass",
    "Description",
    "composition",
    "refractive index",
    "Magnification",
]

xb = np.ascontiguousarray(np.asarray(Z, dtype=np.float32))
q = int(QUERY_ROW) % int(N)
search_k = min(int(K_BAYES) + 1, int(N))
dist2, idx = index.search(xb[q:q+1], search_k)

nbr = []
for ix, d2 in zip(idx[0].tolist(), dist2[0].tolist()):
    if ix < 0 or ix == q:
        continue
    nbr.append((int(ix), float(d2)))
    if len(nbr) >= int(K_BAYES):
        break
if not nbr:
    raise RuntimeError("No FAISS neighbors found for Bayesian table.")

d = np.array([x[1] for x in nbr], dtype=np.float64)
med = np.median(d[d > 0]) if np.any(d > 0) else (np.mean(d) if np.mean(d) > 0 else 1.0)
tau = max(1e-9, float(TAU_FRAC) * float(med))
w = np.exp(-((d / tau) ** 2) / max(float(NEIGHBOR_TEMP), 1e-9))
if len(w) > 0:
    w[0] *= float(TOP1_BOOST)

def _row_cell(i: int, col: str) -> str:
    if col not in _df_manifest.columns:
        return ""
    v = _df_manifest.iloc[int(i)][col]
    if isinstance(v, float) and np.isnan(v):
        return ""
    return str(v).strip()

records = []
for field in FIELD_COLUMNS:
    vocab = sorted({
        str(x).strip() for x in _df_manifest[field].fillna("").astype(str).tolist()
        if str(x).strip() != ""
    }) if field in _df_manifest.columns else []

    mass = defaultdict(float)
    for (ri, _), wi in zip(nbr, w):
        val = _row_cell(ri, field)
        if val:
            mass[val] += float(wi)

    if not vocab:
        vocab = sorted(mass.keys())
    if not vocab:
        records.append({"field": field, "actual": _row_cell(q, field), "top": "", "top_prob": 0.0})
        continue

    prior = float(alpha0) / float(len(vocab))
    denom = float(alpha0) + float(sum(mass.get(v, 0.0) for v in vocab))
    post = {v: (prior + mass.get(v, 0.0)) / max(denom, 1e-12) for v in vocab}
    top_val, top_p = max(post.items(), key=lambda kv: kv[1]) if post else ("", 0.0)
    records.append({"field": field, "actual": _row_cell(q, field), "top": top_val, "top_prob": float(top_p)})

bayes_df = pd.DataFrame(records)
bayes_df["top_prob"] = bayes_df["top_prob"].map(lambda x: f"{x:.3f}")

query_img_name = str(filenames[q]).strip() if q < len(filenames) else ""
query_img_path = IMAGES_DIR / query_img_name if query_img_name else None

fig, (ax_img, ax_tbl) = plt.subplots(
    1, 2,
    figsize=(16, max(4, 0.65 * len(bayes_df) + 2)),
    gridspec_kw={"width_ratios": [1.0, 1.6]},
)

ax_img.axis("off")
if query_img_path and Path(query_img_path).is_file():
    try:
        im = Image.open(query_img_path).convert("RGB")
        ax_img.imshow(im)
        ax_img.set_title(f"Query image (row {q})", fontsize=11)
    except Exception:
        ax_img.text(0.5, 0.5, "query image load error", ha="center", va="center")
else:
    ax_img.text(0.5, 0.5, "query image missing", ha="center", va="center")

ax_tbl.axis("off")
ax_tbl.set_title("Bayesian inference table (student z neighbors)", fontsize=11, pad=8)
col_labels = ["field", "actual", "top", "top_prob"]
cell_text = bayes_df[col_labels].astype(str).values.tolist()
col_widths = [0.18, 0.35, 0.35, 0.12]

tbl = ax_tbl.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellLoc="left",
    colLoc="left",
    colWidths=col_widths,
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.0, 1.35)

plt.tight_layout()
plt.show()
print(f"Bayesian query row={q} | neighbors used={len(nbr)} | fields={len(FIELD_COLUMNS)}")

In [ ]:
# GENERATIVE-STYLE SPECIMEN SUMMARY (deterministic; uses FAISS neighborhood from above).
import re
from collections import Counter
from IPython.display import Markdown, display

SUMMARY_NEIGHBORS = 6

if "q" not in globals() or "nbr" not in globals():
    raise RuntimeError("Run the FAISS and Bayesian cells first (needs q and nbr).")


def _row_text(i: int, col: str) -> str:
    if col not in _df_manifest.columns:
        return ""
    v = _df_manifest.iloc[int(i)][col]
    if isinstance(v, float) and np.isnan(v):
        return ""
    return str(v).strip()


def _tokenize(s: str):
    return re.findall(r"[a-zA-Z][a-zA-Z0-9\-]{2,}", s.lower())


STOP = {
    "the", "and", "with", "for", "from", "this", "that", "these", "those", "into", "over", "under",
    "are", "was", "were", "is", "be", "been", "being", "has", "have", "had", "not", "but", "can",
    "may", "also", "very", "more", "less", "than", "then", "when", "where", "which", "while", "within",
    "without", "shows", "show", "seen", "image", "specimen", "sample", "microscopy", "description",
}

_desc_col = next((c for c in ("microscopy description", "Extended description", "Description") if c in _df_manifest.columns), "Description")
query_desc = _row_text(q, _desc_col)
neigh_rows = [ri for ri, _ in nbr[: int(SUMMARY_NEIGHBORS)]]
neigh_desc = [_row_text(ri, _desc_col) for ri in neigh_rows]

texts = [t for t in [query_desc] + neigh_desc if t]
if not texts:
    raise RuntimeError("No description text found for query/neighbors.")

tokens = []
for t in texts:
    tokens.extend([w for w in _tokenize(t) if w not in STOP])
kw = [w for w, _ in Counter(tokens).most_common(14)]

snippets = []
if query_desc:
    snippets.append(("query", query_desc))
for i, t in enumerate(neigh_desc, start=1):
    if t:
        snippets.append((f"neighbor {i}", t))
snippets = snippets[:6]

spec_name = _row_text(q, "Specimen Name") or "(unnamed specimen)"
desc_name = _row_text(q, "Description") or "(no Description)"
illum = _row_text(q, "Illumination Modality") or "(illumination not specified)"

theme = ", ".join(kw[:8]) if kw else "no dominant recurring terms"

summary = (
    f"For specimen **{spec_name}** (Description: **{desc_name}**, Illumination: **{illum}**), "
    f"the aggregated descriptions from the query and nearest student-z neighbors suggest a consistent visual profile. "
    f"Dominant recurring features include: {theme}. "
    f"Across the local neighborhood, the text points to a coherent morphology/material signature rather than conflicting descriptions."
)

md = ["### Generative summary (student z neighborhood, description aggregate)", "", summary, "", "**Representative evidence snippets**"]
for tag, txt in snippets:
    short = txt if len(txt) <= 240 else (txt[:237] + "...")
    md.append(f"- **{tag}**: {short}")

display(Markdown("\n".join(md)))

## Step 7 — Generative relabel (HKU LLM gateway)

Re-uses the `Z` / FAISS / Bayesian setup above to pull, **per row**, the top-K nearest neighbors and a Dirichlet-smoothed posterior over `logitb`/`Class`/`Subclass`. Sends a strict-JSON prompt to the **HKU Azure-style API** (`api.hku.hk`, default deployment `gpt-5.5`) and writes one JSONL record per row.

This is where `microscopy_description` and `Extended_description` finally earn their keep — they're injected into the prompt as **retrieval-time evidence** rather than features that smear the manifold.

**Cells in this section**
1. **Setup** — resolve API key, build `RelabelConfig` from the in-memory `Z`. No network calls.
2. **Pilot (3 rows)** — full prompt + response printed inline so you can sanity-check before fanning out. Costs <\$0.10 on `gpt-5.5`.
3. **Pilot (50 rows)** — runs the next 50 rows with concurrency=2; reports tokens + per-row time. Costs ~\$1.
4. **Full run** — uses `--filter low_confidence_only` by default (only rows where the student's `y_pred_prob` is below threshold). Resumable: re-running skips rows whose `prompt_hash` matches.
5. **Merge** — writes a `Material_Database_refined.xlsx` audit copy with new `refined_*` columns.

**Pre-requisites:** Run cells 1–4 above first (so `Z`, `MANIFEST_KEYS`, `pred_df`, `EMB_OUT_DIR` are in memory). Set `HKU_API_KEY` either as an env var or in `~/.nehm/hku_api_key`.

In [ ]:
# 7.1 — LLM relabel setup. Builds the RelabelConfig from the in-memory Z + sheet, resolves
# API key. No network calls happen until the next cell.
import importlib
import os
from pathlib import Path

# HKU gateway key wired in directly (matches api-gateway-test.ipynb).
# NOTE: don't commit this notebook to a public repo while the key is hardcoded.
os.environ["HKU_API_KEY"] = "bbb7da6b3cda4159a1a1e35af568923b"

import nehm_pipeline.llm_gateway as _llmgw
import nehm_pipeline.generative_relabel as _grl
importlib.reload(_llmgw)
importlib.reload(_grl)
from nehm_pipeline.generative_relabel import RelabelConfig, run_relabel, merge_jsonl_into_xlsx

RELABEL_OUT_DIR = EMB_OUT_DIR.parent / "generative_relabel"
RELABEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
RELABEL_JSONL = RELABEL_OUT_DIR / "relabel.jsonl"
RELABEL_REFINED_XLSX = RELABEL_OUT_DIR / "Material_Database_refined.xlsx"

# Default deployment; switch to "gpt-4o" / "chatgpt-4" if you want to A/B compare.
LLM_ENGINE = "gpt-5.5"

# Either set HKU_API_KEY in your shell, or save the key once to ~/.nehm/hku_api_key.
import os
_api_key_present = bool(os.environ.get("HKU_API_KEY")) or (Path.home() / ".nehm" / "hku_api_key").is_file()
print(f"HKU API key resolvable : {_api_key_present}")
print(f"  (env HKU_API_KEY    = {'set' if os.environ.get('HKU_API_KEY') else 'unset'})")
print(f"  (~/.nehm/hku_api_key = {(Path.home() / '.nehm' / 'hku_api_key').is_file()})")
print()

# Build the config off the in-memory student Z (no disk reload needed). Image Index alignment
# is verified inside run_relabel.
RELABEL_CFG = RelabelConfig(
    project_root=PROJECT_ROOT,
    xlsx_path=MATERIAL_DB_XLSX,
    sheet_name="Sheet2",
    student_z_array=Z,                   # in-memory 2304-D from cell 4 (B v3, LongCLIP-B)
    student_predictions_csv=STUDENT_PRED_CSV,
    retrieval_space="student_z",
    k_neighbors=10,
    llm_engine=LLM_ENGINE,
    llm_api_key_env="HKU_API_KEY",
    temperature=0.2,
    max_output_tokens=1500,
    concurrency=2,
    min_request_interval_s=0.6,
    output_jsonl=RELABEL_JSONL,
    audit_dir=RELABEL_OUT_DIR,
    dry_run_first_n=0,                   # we drive the dry preview from a separate cell
)
RELABEL_CFG.resolve()
print(f"Z (in-memory)        : shape={Z.shape}")
print(f"output JSONL         : {RELABEL_JSONL}")
print(f"refined xlsx (later) : {RELABEL_REFINED_XLSX}")
print(f"engine               : {LLM_ENGINE}")


In [ ]:
# 7.2 — LLM relabel preview. By DEFAULT this examines the same `QUERY_ROW` you set in the
# FAISS / Bayesian cells above (cells 8 and 9). Just re-run this cell whenever you change
# QUERY_ROW. **No JSONL is written**; the response is printed inline only.
import importlib
import json as _json
import logging as _logging
import nehm_pipeline.llm_gateway as _llmgw
import nehm_pipeline.generative_relabel as _grl
importlib.reload(_llmgw)
importlib.reload(_grl)
from nehm_pipeline.generative_relabel import preview_relabel

_logging.basicConfig(level=_logging.INFO, format="%(asctime)s %(levelname)s | %(message)s", force=True)

# ----- WHAT TO EXAMINE -------------------------------------------------------------------
# DEFAULT path: examine the same QUERY_ROW you set in the FAISS / Bayes cells above.
# Override either by:
#   - adding rows to PILOT_EXTRA_QUERY_ROWS  (row indices, like QUERY_ROW)
#   - adding rows to PILOT_IMAGE_INDICES     (`Image Index` values from the spreadsheet)
#
# IDENTIFICATION MODE
# By default RELABEL_CFG.assume_unknown_query=True: the target's identity-level metadata
# (Class, Subclass, descriptions, composition, ...) is HIDDEN from the LLM, only the
# capture-level metadata (Magnification, Illumination Modality, Stain) is shown. Set
# `_pilot_cfg.assume_unknown_query = False` below to reveal everything (sanity check on
# known specimens).
PILOT_EXTRA_QUERY_ROWS: list[int] = []        # e.g. [1021, 4218]
PILOT_IMAGE_INDICES:    list[int] = []        # e.g. [4271, 1882, 3960]

PILOT_PRINT_PROMPT = True                     # set False once you trust the prompt structure
PILOT_PROMPT_PREVIEW_CHARS = 6000             # bumped — neighbor block (with Extended description) is long
# -----------------------------------------------------------------------------------------

if "QUERY_ROW" not in globals() and not PILOT_EXTRA_QUERY_ROWS and not PILOT_IMAGE_INDICES:
    raise NameError(
        "QUERY_ROW is not defined. Either run the FAISS / Bayes cells above first (which set "
        "QUERY_ROW = ...), or fill PILOT_IMAGE_INDICES / PILOT_EXTRA_QUERY_ROWS in this cell."
    )

_pilot_cfg = RelabelConfig(**{**RELABEL_CFG.__dict__,
    "row_filter": "all",
    "concurrency": 1,
    "dry_run_first_n": 0,
})
_pilot_cfg.resolve()

# Build the row-index list. Order: explicit Image Indices first (resolved), then
# QUERY_ROW + any PILOT_EXTRA_QUERY_ROWS (already row indices, just modulo N).
_resolved_row_indices: list[int] = []

if PILOT_IMAGE_INDICES:
    _df_for_lookup = pd.read_excel(_pilot_cfg.xlsx_path, sheet_name=_pilot_cfg.sheet_name)
    _df_for_lookup = _df_for_lookup.sort_values(_pilot_cfg.image_index_col).reset_index(drop=True)
    _ii_to_row = {int(v): int(i) for i, v in enumerate(_df_for_lookup[_pilot_cfg.image_index_col].astype(int).tolist())}
    _missing = []
    for _ii in PILOT_IMAGE_INDICES:
        if int(_ii) in _ii_to_row:
            _resolved_row_indices.append(_ii_to_row[int(_ii)])
        else:
            _missing.append(int(_ii))
    if _missing:
        print(f"WARNING: not found in spreadsheet (skipped): {_missing}")

if "QUERY_ROW" in globals() and not PILOT_IMAGE_INDICES:
    _resolved_row_indices.append(int(QUERY_ROW) % int(N))

for _r in PILOT_EXTRA_QUERY_ROWS:
    _resolved_row_indices.append(int(_r) % int(N))

# de-dupe while preserving order
_seen = set()
_resolved_row_indices = [r for r in _resolved_row_indices if not (r in _seen or _seen.add(r))]

if not _resolved_row_indices:
    raise ValueError("No valid rows resolved. Set QUERY_ROW above or populate the override lists.")

print(f"examining {len(_resolved_row_indices)} row(s): row_idx={_resolved_row_indices}")
print()

# --- Diagnostic: confirm which sheet columns are wired into the prompts ---
_diag_df = pd.read_excel(_pilot_cfg.xlsx_path, sheet_name=_pilot_cfg.sheet_name)
_diag_target = _pilot_cfg.target_capture_fields if _pilot_cfg.assume_unknown_query else _pilot_cfg.field_columns
_diag_target_present = [c for c in _diag_target if c in _diag_df.columns]
_diag_target_missing = [c for c in _diag_target if c not in _diag_df.columns]
_diag_neighbor_fields = [c for c in _pilot_cfg.field_columns if c in _diag_df.columns]
_diag_neighbor_missing = [c for c in _pilot_cfg.field_columns if c not in _diag_df.columns]
_diag_text_fields = [c for c in _pilot_cfg.neighbor_text_fields if c in _diag_df.columns]
print(f"assume_unknown_query : {_pilot_cfg.assume_unknown_query}")
print(f"target fields shown  : {_diag_target_present}")
if _diag_target_missing:
    print(f"  WARNING target columns missing from sheet: {_diag_target_missing}")
print(f"neighbor fields shown: {_diag_neighbor_fields}")
if _diag_neighbor_missing:
    print(f"  WARNING neighbor columns missing from sheet: {_diag_neighbor_missing}")
print(f"neighbor text blocks : {_diag_text_fields}  (full text, not key=value)")
print()

_results = preview_relabel(_pilot_cfg, row_indices=_resolved_row_indices)

for _i, _r in enumerate(_results, 1):
    print("=" * 78)
    print(f"  ROW {_i}/{len(_results)}  Image Index = {_r['image_index']}  (row_idx={_r['row_idx']})")
    print("=" * 78)
    print(f"  neighbors (Image Index) : {_r['neighbor_image_indices']}")
    print(f"  weights                 : " + ", ".join(f"{w:.3f}" for w in _r['neighbor_weights']))
    print(f"  posteriors_top3         :")
    for _col, _items in _r["posteriors_top3"].items():
        print(f"    {_col}: " + " | ".join(f"{n} (p={p:.3f})" for n, p in _items))

    if PILOT_PRINT_PROMPT:
        print("\n  --- USER PROMPT (truncated) ---")
        _up = _r["user_prompt"]
        print(_up[:PILOT_PROMPT_PREVIEW_CHARS] + ("\n  …(truncated)" if len(_up) > PILOT_PROMPT_PREVIEW_CHARS else ""))

    print("\n  --- RESPONSE (parsed JSON) ---")
    print(_json.dumps(_r["response"], indent=2, ensure_ascii=False))
    print(f"\n  usage so far: {_r['usage_after_call']}")
    print()


# Step 8 — Lasso + Approximate K-SVD on the student z (2304-D, B v3)

This section ports `Painting_Interpretability_Study/Stable/Notebooks/Student_Embed_KSVD.ipynb`
to the NEHM Pipeline B v3 student z. It learns a sparse, signed dictionary

```
Y ≈ D @ W
```

where `Y` is `(features, samples) = (2304, N)`, `D` is `(2304, n_atoms)`, and `W` is
`(n_atoms, N)`. The loop alternates **OMP / Lasso sparse coding** with a fast **approximate
K-SVD residual update** on each atom.

The 2304-D student z naturally splits into the **four teacher blocks**
`[image_512 ⨁ illum_512 ⨁ class+subclass_512 ⨁ desc+meta_512]`, so each learned atom has a
clean **block-energy footprint**: how much of the atom's direction lives in the image branch,
the illumination branch, the class/subclass text branch, and the long-description text branch.

Each atom can also be passed through the frozen student `classifier` to get a **K-D logit
footprint** — i.e. which `logitb` weak-label classes that atom direction pushes toward.

**What you get out**

- `D` `(2304, n_atoms)` — learned dictionary atoms in standardized z-space.
- `C_all` `(N, n_atoms)` — per-row sparse codes.
- `atom_usage.csv` — frequency / mean-abs / max-abs activation per atom.
- `atom_block_footprints.csv` — normalized block energy per atom across the 4 teacher blocks.
- `atom_logit_deltas.npy` — `(n_atoms, K)` classifier response to each atom direction.
- Diagnostic plots: reconstruction overlay, atom shape line plots, block-energy heatmap,
  per-sample active-atom panels, FAISS-like top-9 similar-by-pattern thumbnails.
- A small **Bayesian metadata inference** stage that, for any atom, builds a kernel-weighted
  posterior over `Class`, `Subclass`, `Illumination Modality`, and free-text Description tokens
  from the top-K samples where the atom is most expressed.

In [ ]:
# 8.0 — K-SVD config. Source = student z (2304-D) loaded above as `Z` / `z_all`.
import json

from sklearn.decomposition import sparse_encode
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

KSVD_SEED = 42
ksvd_rng = np.random.default_rng(KSVD_SEED)

KSVD_OUT_DIR = EMB_OUT_DIR / "ksvd_dictionary"
KSVD_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Atom count rule of thumb: ~10-20× over the input dim. With z=2304 we use 400 to keep the
# per-iteration cost manageable on CPU; bump to 800-1200 for finer-grained atom semantics.
N_ATOMS = 400
KSVD_OUTER_ITERS = 30

# Sparse coding: "omp" gives a fixed sparsity per row (recommended for K-SVD); "lasso" gives
# alpha-controlled sparsity (use when you want different rows to have very different actives).
SPARSE_CODE_METHOD = "omp"
SPARSE_CODE_N_NONZERO_COEFS = 30
LASSO_ALPHA = 0.5
LASSO_ALGORITHM = "lasso_cd"

SUBSAMPLE_FRACTION = 1.0
ACTIVE_EPS = 1e-2
REINIT_DEAD_ATOMS = True
MIN_DICT_CHANGE = 1e-5
PATIENCE = 3

# Teacher block layout: 768-D image + 3×512-D text (pre-projection).
BLOCK_NAMES = ["image", "illumination", "class_subclass", "desc_meta"]
BLOCK_DELIMS = [
    EXPECTED_IMAGE_DIM,
    EXPECTED_IMAGE_DIM + EXPECTED_TEXT_BLOCK_DIM,
    EXPECTED_IMAGE_DIM + 2 * EXPECTED_TEXT_BLOCK_DIM,
]  # vertical guides at 768, 1280, 1792
TEACHER_DIM = EXPECTED_TEACHER_DIM  # 2304

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 10

print(f"KSVD_OUT_DIR        : {KSVD_OUT_DIR}")
print(f"N_ATOMS             : {N_ATOMS}")
print(f"KSVD_OUTER_ITERS    : {KSVD_OUTER_ITERS}")
print(f"sparse coding       : {SPARSE_CODE_METHOD} (n_nonzero={SPARSE_CODE_N_NONZERO_COEFS}, lasso_alpha={LASSO_ALPHA})")
print(f"block layout        : {BLOCK_NAMES} → [768, 512, 512, 512] = {TEACHER_DIM}-D total")

## 8.1 — Standardize the student z

The dictionary is learned on the L2-normalized 2304-D student z, **standardized to zero mean
and unit variance per dimension** (sklearn `StandardScaler` fit on a held-out 85% split). This
keeps each of the four 512-D teacher blocks on roughly the same scale so no single block
hogs the sparse code. The val-15% split is used only for an honest reconstruction-MSE
diagnostic.

In [ ]:
# 8.2 — Load + standardize student z. Reuses `Z` / `z_all` from cell 4 (or reads from disk).
if "z_all" in globals():
    student_z = np.asarray(z_all, dtype=np.float32)
else:
    student_z = np.load(STUDENT_Z_NPY).astype(np.float32)

if student_z.ndim != 2 or student_z.shape[1] != TEACHER_DIM:
    raise ValueError(
        f"student_z shape {student_z.shape} is not (N, {TEACHER_DIM}); "
        "expected the B v3 (2304-D LongCLIP-B fused teacher) student output."
    )

# logits live alongside z; load if missing so the per-atom logit footprint cell can run.
if "logits_all" in globals():
    student_logits = np.asarray(logits_all, dtype=np.float32)
else:
    student_logits = np.load(STUDENT_LOGITS_NPY).astype(np.float32)

idx_all = np.arange(student_z.shape[0])
train_idx, val_idx = train_test_split(idx_all, test_size=0.15, random_state=KSVD_SEED, shuffle=True)

ksvd_scaler = StandardScaler()
Z_train_std = ksvd_scaler.fit_transform(student_z[train_idx]).astype(np.float32)
Z_val_std   = ksvd_scaler.transform(student_z[val_idx]).astype(np.float32)
Z_all_std   = ksvd_scaler.transform(student_z).astype(np.float32)

# K-SVD math expects (features, samples), not (samples, features).
Y_train = Z_train_std.T.astype(np.float32)
Y_val   = Z_val_std.T.astype(np.float32)
Y_all   = Z_all_std.T.astype(np.float32)

print(f"student_z (raw)    : {student_z.shape}")
print(f"student_logits     : {student_logits.shape}")
print(f"Y_train            : {Y_train.shape}")
print(f"Y_val              : {Y_val.shape}")
print(f"per-dim std (train): mean={float(ksvd_scaler.scale_.mean()):.4f}  "
      f"min={float(ksvd_scaler.scale_.min()):.4f}  max={float(ksvd_scaler.scale_.max()):.4f}")

## 8.3 — K-SVD helpers

Signed sparse coding (OMP or Lasso) followed by a fast approximate K-SVD atom update:

```text
R = Y - D @ W
Ri = R[:, active] + D[:, k] * W[k, active]
dk = Ri @ W[k, active]
D[:, k] = dk / ||dk||
W[k, active] = D[:, k].T @ Ri
```

This is not the full one-atom SVD update from the original Aharon-Elad-Bruckstein paper, but
it preserves the residual-correction structure and runs ~10× faster, which matters because the
student z is ~5,400 × 2304 and the OMP step dominates wall time per iteration.

In [ ]:
# 8.4 — K-SVD helpers (sparse coding + approximate K-SVD update + recon stats).
def normalize_columns(D: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(D, axis=0, keepdims=True)
    return D / np.maximum(norms, 1e-12)


def initial_dictionary_from_samples(Y: np.ndarray, n_atoms: int) -> np.ndarray:
    n_samples = Y.shape[1]
    ix = ksvd_rng.choice(n_samples, size=n_atoms, replace=n_atoms > n_samples)
    return normalize_columns(Y[:, ix].copy()).astype(np.float32)


def lasso_sparse_code(Y: np.ndarray, D: np.ndarray, alpha: float) -> np.ndarray:
    codes = sparse_encode(
        Y.T, D.T, algorithm=LASSO_ALGORITHM, alpha=float(alpha), max_iter=5000,
    )
    return codes.T.astype(np.float32)


def omp_sparse_code(Y: np.ndarray, D: np.ndarray, n_nonzero_coefs: int) -> np.ndarray:
    codes = sparse_encode(
        Y.T, D.T, algorithm="omp", n_nonzero_coefs=int(n_nonzero_coefs),
    )
    return codes.T.astype(np.float32)


def sparse_code(Y: np.ndarray, D: np.ndarray) -> np.ndarray:
    method = str(SPARSE_CODE_METHOD).lower().strip()
    if method == "omp":
        return omp_sparse_code(Y, D, SPARSE_CODE_N_NONZERO_COEFS)
    if method == "lasso":
        return lasso_sparse_code(Y, D, LASSO_ALPHA)
    raise ValueError(f"Unknown SPARSE_CODE_METHOD={SPARSE_CODE_METHOD!r}; use 'omp' or 'lasso'.")


def approximate_ksvd_update(Y: np.ndarray, D: np.ndarray, W: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    R = Y - D @ W
    for k in range(D.shape[1]):
        active = np.flatnonzero(np.abs(W[k]) > ACTIVE_EPS)
        if len(active) == 0:
            if REINIT_DEAD_ATOMS:
                j = int(np.argmax(np.linalg.norm(R, axis=0)))
                atom = R[:, j].copy()
                norm = np.linalg.norm(atom)
                if norm > 1e-12:
                    D[:, k] = atom / norm
            continue
        Ri = R[:, active] + np.outer(D[:, k], W[k, active])
        dk = Ri @ W[k, active].T
        norm = np.linalg.norm(dk)
        if norm <= 1e-12:
            continue
        D[:, k] = dk / norm
        W[k, active] = D[:, k].T @ Ri
        R[:, active] = Ri - np.outer(D[:, k], W[k, active])
    return D.astype(np.float32), W.astype(np.float32)


def reconstruction_stats(Y: np.ndarray, D: np.ndarray, W: np.ndarray) -> dict:
    recon = D @ W
    active = (np.abs(W) > ACTIVE_EPS).sum(axis=0)
    dead = (np.abs(W) > ACTIVE_EPS).sum(axis=1) == 0
    return {
        "mse": float(np.mean((Y - recon) ** 2)),
        "mean_active": float(active.mean()),
        "min_active": int(active.min()),
        "max_active": int(active.max()),
        "dead_atoms": int(dead.sum()),
    }

## 8.5 — Fit the dictionary

Tune `N_ATOMS`, `KSVD_OUTER_ITERS`, `SPARSE_CODE_METHOD`, and `SUBSAMPLE_FRACTION` in the
config cell at the top of this section. Wall time on the ~5,400 × 2304 NEHM data: ~2-5 min on
a recent x86 CPU per outer iter (most of it OMP). Set `KSVD_OUTER_ITERS = 5` for a smoke test.

The loop early-stops if the dictionary changes by less than `MIN_DICT_CHANGE` for `PATIENCE`
consecutive iterations (defaults: `1e-5` for 3 iters).

In [ ]:
# 8.6 — Fit approximate K-SVD on the standardized student z.
D = initial_dictionary_from_samples(Y_train, N_ATOMS)
ksvd_history = []
small_change_count = 0

for _it in range(1, KSVD_OUTER_ITERS + 1):
    D_prev = D.copy()
    if SUBSAMPLE_FRACTION < 1.0:
        n_sub = max(N_ATOMS, int(round(Y_train.shape[1] * SUBSAMPLE_FRACTION)))
        sub_ix = ksvd_rng.choice(Y_train.shape[1], size=min(n_sub, Y_train.shape[1]), replace=False)
        Y_iter = Y_train[:, sub_ix]
    else:
        Y_iter = Y_train

    W_iter = sparse_code(Y_iter, D)
    D, W_iter = approximate_ksvd_update(Y_iter, D, W_iter)
    D = normalize_columns(D).astype(np.float32)

    stats = reconstruction_stats(Y_iter, D, W_iter)
    dict_change = float(np.linalg.norm(D - D_prev) / max(np.linalg.norm(D_prev), 1e-12))
    row = {"iteration": _it, "samples": int(Y_iter.shape[1]), "dict_change": dict_change, **stats}
    ksvd_history.append(row)
    print(
        f"iter {_it:03d}/{KSVD_OUTER_ITERS} | mse={stats['mse']:.6f} | "
        f"active={stats['mean_active']:.1f} ({stats['min_active']}-{stats['max_active']}) | "
        f"dead={stats['dead_atoms']} | d_change={dict_change:.6e}"
    )
    if dict_change < MIN_DICT_CHANGE:
        small_change_count += 1
        if small_change_count >= PATIENCE:
            print(f"Early stop after {_it} iterations: dictionary change < {MIN_DICT_CHANGE}")
            break
    else:
        small_change_count = 0

ksvd_history_df = pd.DataFrame(ksvd_history)
ksvd_history_df.to_csv(KSVD_OUT_DIR / "ksvd_training_history.csv", index=False)
print(f"D: {D.shape}  | history written to {KSVD_OUT_DIR / 'ksvd_training_history.csv'}")
ksvd_history_df.tail()

## 8.7 — Sparse-code every row + atom-usage table

Compute sparse codes for **all** rows under the learned dictionary. `C_all` has shape
`(N, n_atoms)` and is what every downstream cell (sample inspection, similarity, Bayesian
posterior) keys off.

In [ ]:
# 8.8 — Sparse-code all rows; build atom-usage table.
W_all = sparse_code(Y_all, D)
Z_recon_std = (D @ W_all).T.astype(np.float32)
C_all = W_all.T.astype(np.float32)
student_z_recon = ksvd_scaler.inverse_transform(Z_recon_std).astype(np.float32)

W_val_codes = sparse_code(Y_val, D)
Z_val_recon_std = (D @ W_val_codes).T.astype(np.float32)

mse_all_std = float(np.mean((Z_all_std - Z_recon_std) ** 2))
mse_val_std = float(np.mean((Z_val_std - Z_val_recon_std) ** 2))
active_per_sample = (np.abs(C_all) > ACTIVE_EPS).sum(axis=1)

usage_count = (np.abs(C_all) > ACTIVE_EPS).sum(axis=0)
mean_abs_code = np.mean(np.abs(C_all), axis=0)
max_abs_code = np.max(np.abs(C_all), axis=0)
atom_usage = pd.DataFrame(
    {
        "atom": np.arange(C_all.shape[1]),
        "usage_count": usage_count,
        "usage_fraction": usage_count / C_all.shape[0],
        "mean_abs_code": mean_abs_code,
        "max_abs_code": max_abs_code,
    }
).sort_values("mean_abs_code", ascending=False)
atom_usage.to_csv(KSVD_OUT_DIR / "atom_usage.csv", index=False)

print(f"all standardized recon MSE: {mse_all_std:.6f}")
print(f"val standardized recon MSE: {mse_val_std:.6f}")
print(f"active atoms/sample      : mean={active_per_sample.mean():.2f}  "
      f"min={active_per_sample.min()}  max={active_per_sample.max()}")
print(f"dead atoms                : {(usage_count == 0).sum()}")
display(atom_usage.head(20))

## 8.9 — Reconstruction overlay for a single sample

Plots the original standardized z and its sparse reconstruction on shared axes, plus the
residual. Vertical guides at `512`, `1024`, `1536` mark the four teacher-block boundaries —
useful for spotting which block(s) the reconstruction handles cleanly vs which it leaves on
the table. Below the plot is a ranked table of the top-15 active atoms for that sample.

In [ ]:
# 8.10 — Reconstruction overlay for one sample.
KSVD_RECON_SAMPLE = min(1387, student_z.shape[0] - 1)
orig = Z_all_std[KSVD_RECON_SAMPLE]
recon = Z_recon_std[KSVD_RECON_SAMPLE]
resid = orig - recon
active_atoms_sample = np.flatnonzero(np.abs(C_all[KSVD_RECON_SAMPLE]) > ACTIVE_EPS)

y_min = float(min(orig.min(), recon.min()))
y_max = float(max(orig.max(), recon.max()))
y_pad = 0.05 * max(y_max - y_min, 1e-6)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(orig, linewidth=1.1, label="original", alpha=0.9)
axes[0].plot(recon, linewidth=1.1, label="reconstruction", alpha=0.9)
for x_div in BLOCK_DELIMS:
    axes[0].axvline(x_div, color="black", linewidth=0.7, alpha=0.4)
axes[0].set_ylim(y_min - y_pad, y_max + y_pad)
axes[0].set_title(
    f"Overlay: original vs K-SVD recon | sample {KSVD_RECON_SAMPLE} "
    f"(image_idx={int(MANIFEST_KEYS[KSVD_RECON_SAMPLE])}) | active atoms: {len(active_atoms_sample)}"
)
axes[0].set_ylabel("standardized value")
axes[0].legend(loc="upper right")
axes[0].grid(alpha=0.2)

axes[1].plot(resid, linewidth=1, color="tab:red")
for x_div in BLOCK_DELIMS:
    axes[1].axvline(x_div, color="black", linewidth=0.7, alpha=0.4)
axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.5)
axes[1].set_title(f"Residual | MSE={float(np.mean(resid ** 2)):.6f}  "
                  f"(block boundaries marked at 768 / 1280 / 1792)")
axes[1].set_xlabel("student z dimension")
axes[1].set_ylabel("orig - recon")
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

top = np.argsort(np.abs(C_all[KSVD_RECON_SAMPLE]))[::-1][:15]
display(pd.DataFrame({
    "rank": np.arange(1, len(top) + 1),
    "atom": top,
    "weight": C_all[KSVD_RECON_SAMPLE, top],
    "abs_weight": np.abs(C_all[KSVD_RECON_SAMPLE, top]),
}))

## 8.11 — Atom shape line plots (shared y-scale)

Eight randomly chosen atoms drawn on a shared symmetric y-scale so you can read off relative
amplitudes. Vertical guides mark the four teacher blocks.

In [ ]:
# 8.12 — Atom shape line plots (shared y-scale).
n_show = min(8, D.shape[1])
atom_ids = ksvd_rng.choice(D.shape[1], size=n_show, replace=False)
selected_atoms = D[:, atom_ids].T
y_lim = max(float(np.max(np.abs(selected_atoms))) * 1.05, 1e-6)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.0 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]
for ax, atom_id in zip(axes, atom_ids):
    atom = D[:, atom_id]
    ax.plot(atom, linewidth=1)
    for x_div in BLOCK_DELIMS:
        ax.axvline(x_div, color="black", linewidth=0.7, alpha=0.35)
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.5)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_title(f"Atom {atom_id} | norm={np.linalg.norm(atom):.3f} | shared y-scale +/-{y_lim:.4f}")
    ax.grid(alpha=0.2)
axes[-1].set_xlabel("student z dimension")
plt.tight_layout()
plt.show()

## 8.13 — Decode atoms through the frozen student classifier

Each atom is a 2304-D direction in standardized z-space. We can ask **what classifier
response that direction produces** by mapping each atom back to the original z-scale (via the
inverse of `StandardScaler`) and then evaluating

```text
logit_delta(atom) = student.classifier(z_mean + atom_raw) - student.classifier(z_mean)
```

This gives a `(n_atoms, K)` matrix of per-atom logit deltas — i.e. which weak-label classes
each atom direction pushes toward (or away from). Same idea as the painting notebook's
"decode atoms through finetune-3 head", but here the head is just `student.classifier` since
the atoms already live in z-space.

In [ ]:
# 8.14 — Decode each atom through student.classifier → (n_atoms, K) logit deltas.
ATOM_FOOTPRINT_SCALE = 1.0

z_mean = ksvd_scaler.mean_.astype(np.float32)
z_scale = ksvd_scaler.scale_.astype(np.float32)
atom_deltas_raw = (D.T * z_scale[None, :]).astype(np.float32)
atom_inputs_raw = (z_mean[None, :] + ATOM_FOOTPRINT_SCALE * atom_deltas_raw).astype(np.float32)

# `student.classifier` is the frozen `Linear(teacher_dim → K)` head trained jointly with the
# decoder. Run it on baseline + each atom to get the relative logit footprint.
with torch.inference_mode():
    baseline_in = torch.from_numpy(z_mean[None, :]).to(DEVICE)
    baseline_logits = student.classifier(baseline_in).detach().cpu().numpy().astype(np.float32)
    atom_in = torch.from_numpy(atom_inputs_raw).to(DEVICE)
    atom_logits_raw = student.classifier(atom_in).detach().cpu().numpy().astype(np.float32)

atom_logit_delta = (atom_logits_raw - baseline_logits).astype(np.float32)

np.save(KSVD_OUT_DIR / "atom_z_deltas_raw.npy", atom_deltas_raw)
np.save(KSVD_OUT_DIR / "atom_logits.npy", atom_logits_raw)
np.save(KSVD_OUT_DIR / "atom_logit_deltas.npy", atom_logit_delta)
print(f"atom_logit_delta : {atom_logit_delta.shape}  -> {KSVD_OUT_DIR / 'atom_logit_deltas.npy'}")

## 8.15 — Inspect the active atoms of a *single* image

`C_all` (shape `(N, n_atoms)`) is the per-row sparse code matrix produced by the K-SVD fit
above. Each row tells you which dictionary atoms the student's 2304-D `z` decomposes onto for
one specimen, and the *signed* weight of each atom.

This cell picks one row (`KSVD_QUERY_SAMPLE`, default = the FAISS `QUERY_ROW` from the upstream
neighbor panel) and renders three things:

1. The **full sparse code** as a signed bar chart over all `n_atoms` (block boundaries at 512 /
   1024 / 1536 are not relevant here — the bars are *atoms*, not z-dimensions).
2. The **ranked active atoms only** (`|weight| > ACTIVE_EPS`), sorted by `abs(weight)`, so you
   can tell at a glance which atoms drive this specimen.
3. A **per-atom decode table**: for every active atom, the table joins (a) its `atom_usage`
   stats from cell 8.8, (b) its `atom_logit_delta` top + bottom classifier classes from
   cell 8.14, and (c) the `tree_class_key` cohort of the rows where the atom fires hardest, so
   you can read off "what this atom expresses" without leaving the cell.

In [ ]:
# 8.16 — Per-image active atoms + per-atom decode table.
# Default sample = the FAISS QUERY_ROW chosen for the image-space neighbor panel above; override
# KSVD_QUERY_SAMPLE here to inspect any other row without rerunning the whole pipeline.
KSVD_QUERY_SAMPLE       = int(globals().get("QUERY_ROW", KSVD_RECON_SAMPLE)) % int(C_all.shape[0])
KSVD_QUERY_ACTIVE_EPS   = float(ACTIVE_EPS)
KSVD_QUERY_TOP_N        = None    # None = show all atoms above ACTIVE_EPS; or an int (e.g. 20)
KSVD_QUERY_TOP_LOGIT_K  = 3       # top + bottom classes from atom_logit_delta to surface per atom
KSVD_QUERY_COHORT_TOPK  = 12      # n top-firing samples used to summarize per-atom metadata

# ---- Resolve class names once (used for the "what classes does this atom push" table) ----
_class_names_meta = None
if isinstance(_ckpt_meta, dict):
    _class_names_meta = (_ckpt_meta.get("meta") or {}).get("class_names")
if _class_names_meta is not None and len(_class_names_meta) == atom_logit_delta.shape[1]:
    KSVD_CLASS_NAMES = [str(c) for c in _class_names_meta]
elif "tree_class_key" in _df_manifest.columns and MATERIAL_LOGIT_COLUMN in _df_manifest.columns:
    _label_to_key = (
        _df_manifest[[MATERIAL_LOGIT_COLUMN, "tree_class_key"]]
        .dropna(subset=[MATERIAL_LOGIT_COLUMN])
        .astype({MATERIAL_LOGIT_COLUMN: "Int64"})
        .groupby(MATERIAL_LOGIT_COLUMN)["tree_class_key"]
        .agg(lambda s: s.dropna().astype(str).mode().iat[0] if not s.dropna().empty else "")
        .to_dict()
    )
    KSVD_CLASS_NAMES = [str(_label_to_key.get(k, f"class_{k}")) for k in range(atom_logit_delta.shape[1])]
else:
    KSVD_CLASS_NAMES = [f"class_{k}" for k in range(atom_logit_delta.shape[1])]

# ---- Active atoms for the chosen sample, sorted by abs(weight) descending ----
sample_code = C_all[KSVD_QUERY_SAMPLE].astype(np.float32)
active_atoms = np.flatnonzero(np.abs(sample_code) > KSVD_QUERY_ACTIVE_EPS)
if active_atoms.size == 0:
    active_atoms = np.argsort(np.abs(sample_code))[::-1][: min(10, sample_code.size)]
    print(f"No atoms above ACTIVE_EPS={KSVD_QUERY_ACTIVE_EPS}; falling back to strongest 10 nonzero candidates.")
active_atoms = active_atoms[np.argsort(np.abs(sample_code[active_atoms]))[::-1]]
if KSVD_QUERY_TOP_N is not None:
    active_atoms = active_atoms[: int(KSVD_QUERY_TOP_N)]

# ---- Per-atom semantic decode: top/bottom classifier classes + dominant tree-class cohort ----
def _topk_class_names(deltas: np.ndarray, k: int, *, descending: bool) -> tuple[list[str], list[float]]:
    if descending:
        ix = np.argsort(deltas)[::-1][:k]
    else:
        ix = np.argsort(deltas)[:k]
    return [KSVD_CLASS_NAMES[i] for i in ix], [float(deltas[i]) for i in ix]


def _atom_metadata_cohort(atom_id: int, *, top_k: int, sign_aware: bool = True) -> dict:
    col = C_all[:, int(atom_id)].astype(np.float32)
    if sign_aware:
        sign = 1.0 if sample_code[int(atom_id)] >= 0 else -1.0
        scores = sign * col
    else:
        scores = np.abs(col)
    order = np.argsort(scores)[::-1][: max(int(top_k), 1)]
    cohort = _df_manifest.iloc[order]
    illum_top = cohort["Illumination Modality"].dropna().astype(str).mode()
    cls_top   = cohort["Class"].dropna().astype(str).mode() if "Class" in cohort.columns else pd.Series(dtype=str)
    sub_top   = cohort["Subclass"].dropna().astype(str).mode() if "Subclass" in cohort.columns else pd.Series(dtype=str)
    desc_top  = cohort["Description"].dropna().astype(str).str.strip().value_counts().head(3).index.tolist() \
                if "Description" in cohort.columns else []
    return {
        "dominant_illum":    illum_top.iat[0] if not illum_top.empty else "",
        "dominant_class":    cls_top.iat[0]   if not cls_top.empty   else "",
        "dominant_subclass": sub_top.iat[0]   if not sub_top.empty   else "",
        "top_descriptions":  ", ".join(desc_top),
    }


_decode_rows = []
for _rank, _a in enumerate(active_atoms, start=1):
    _w = float(sample_code[int(_a)])
    _abs_w = abs(_w)
    _sign = "+" if _w >= 0 else "-"
    _signed_deltas = (_w / max(_abs_w, 1e-12)) * atom_logit_delta[int(_a)]   # flip sign if atom fires negative
    _top_names, _top_deltas = _topk_class_names(_signed_deltas, KSVD_QUERY_TOP_LOGIT_K, descending=True)
    _bot_names, _bot_deltas = _topk_class_names(_signed_deltas, KSVD_QUERY_TOP_LOGIT_K, descending=False)
    _cohort = _atom_metadata_cohort(int(_a), top_k=KSVD_QUERY_COHORT_TOPK, sign_aware=True)
    _decode_rows.append({
        "rank":               _rank,
        "atom":               int(_a),
        "weight":             _w,
        "abs_weight":         _abs_w,
        "sign":               _sign,
        "usage_count":        int((np.abs(C_all[:, int(_a)]) > KSVD_QUERY_ACTIVE_EPS).sum()),
        "top_classes_pushed": ", ".join(f"{n} (Δ={d:+.2f})" for n, d in zip(_top_names, _top_deltas)),
        "top_classes_pulled": ", ".join(f"{n} (Δ={d:+.2f})" for n, d in zip(_bot_names, _bot_deltas)),
        **_cohort,
    })

per_atom_decode_df = pd.DataFrame(_decode_rows)

# ---- Header info for this specimen ----
_sample_file = str(filenames[KSVD_QUERY_SAMPLE]) if KSVD_QUERY_SAMPLE < len(filenames) else f"row_{KSVD_QUERY_SAMPLE}"
def _safe(col):
    if col not in _df_manifest.columns:
        return ""
    v = _df_manifest.iloc[KSVD_QUERY_SAMPLE][col]
    return "" if (isinstance(v, float) and np.isnan(v)) else str(v).strip()
_pred_class_id = int(np.argmax(student_logits[KSVD_QUERY_SAMPLE]))
_pred_class_name = KSVD_CLASS_NAMES[_pred_class_id] if _pred_class_id < len(KSVD_CLASS_NAMES) else f"class_{_pred_class_id}"

print(
    f"row {KSVD_QUERY_SAMPLE} | image_idx={int(MANIFEST_KEYS[KSVD_QUERY_SAMPLE])} | file={_sample_file}"
    f"\n  illum    = {_safe('Illumination Modality')}"
    f"\n  class    = {_safe('Class')}"
    f"\n  subclass = {_safe('Subclass')}"
    f"\n  desc     = {_safe('Description')}"
    f"\n  tree_key = {_safe('tree_class_key')}"
    f"\n  in_group_5 = {_safe('in_group_5')}"
    f"\n  student top-1 = {_pred_class_name} (id={_pred_class_id})"
    f"\n  active atoms (|w|>{KSVD_QUERY_ACTIVE_EPS}) = {len(active_atoms)}"
)
display(per_atom_decode_df)

# ---- Plot: full sparse code (top) + ranked actives only (bottom) ----
fig, axes = plt.subplots(2, 1, figsize=(16, 7), gridspec_kw={"height_ratios": [1, 1.4]})

_all_colors = np.where(sample_code >= 0, "tab:blue", "tab:red")
axes[0].bar(np.arange(C_all.shape[1]), sample_code, color=_all_colors, alpha=0.75, width=1.0)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].axhline(KSVD_QUERY_ACTIVE_EPS,  color="gray", linewidth=0.8, linestyle="--", alpha=0.7)
axes[0].axhline(-KSVD_QUERY_ACTIVE_EPS, color="gray", linewidth=0.8, linestyle="--", alpha=0.7)
axes[0].set_title(f"Sparse atom weights | row {KSVD_QUERY_SAMPLE} | {_sample_file}")
axes[0].set_xlabel("atom index")
axes[0].set_ylabel("signed weight")
axes[0].grid(alpha=0.2)

_rank_colors = np.where(sample_code[active_atoms] >= 0, "tab:blue", "tab:red")
axes[1].bar(np.arange(len(active_atoms)), sample_code[active_atoms], color=_rank_colors)
axes[1].set_xticks(np.arange(len(active_atoms)))
axes[1].set_xticklabels([str(a) for a in active_atoms], rotation=90, fontsize=8)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Active atoms only | active={len(active_atoms)} | threshold={KSVD_QUERY_ACTIVE_EPS}")
axes[1].set_xlabel("atom index, sorted by abs(weight)")
axes[1].set_ylabel("signed weight")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# Persist for downstream cells.
per_atom_decode_df.to_csv(KSVD_OUT_DIR / f"per_atom_decode_row{KSVD_QUERY_SAMPLE}.csv", index=False)
print(f"-> wrote {KSVD_OUT_DIR / f'per_atom_decode_row{KSVD_QUERY_SAMPLE}.csv'}")

## 8.17 — FAISS neighbor lookup in **atom-strength** space

The image-space FAISS panel earlier in this notebook indexes the dense student `z` (2304-D)
and answers the question *"which specimens look alike to the student backbone?"*. This cell
indexes a different signal: the per-row **sparse code** `C_all` (shape `(N, n_atoms)`), so the
question becomes *"which specimens use the same atoms with the same signs and similar
strengths?"*.

The pipeline is the standard FAISS cosine-via-inner-product trick:

1. Optionally hard-threshold each row: keep only weights with `|w| > FAISS_ATOM_EPS`, or only
   the top-`FAISS_ATOM_TOPK` strongest weights per row. This kills the OMP "leakage" that
   would otherwise make every row look weakly similar to every other row.
2. L2-normalize each thresholded row (rows that go all-zero get a score of 0 and are
   excluded).
3. Build a `faiss.IndexFlatIP` and search with the query row's normalized pattern.
4. Inner-product on L2-normalized vectors = cosine similarity, in `[-1, 1]`. Negative scores
   indicate atom patterns that are *anti-correlated* (e.g. opposite signs on the shared
   atoms).

The result is a thumbnail panel + a tabular view that lists the top neighbors, the cosine
similarity, the count of *shared* active atoms, and the strongest shared atom weights, plus
their `(Illumination Modality | Class | Subclass | Description)` tree-class metadata so you can
see at a glance whether the atom-pattern neighbors agree with the curator labels.

In [ ]:
# 8.18 — FAISS index over sparse codes; query by atom-strength fingerprint.
# The image-space FAISS panel above used IndexFlatL2 on dense z. Here we use IndexFlatIP on
# L2-normalized sparse codes so inner-product == cosine similarity over atom patterns.
import subprocess as _subprocess_ksvd
import sys as _sys_ksvd

try:
    import faiss  # type: ignore
except ModuleNotFoundError:
    print("Installing faiss-cpu ...", flush=True)
    _subprocess_ksvd.check_call([_sys_ksvd.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
    import faiss  # type: ignore

FAISS_ATOM_QUERY      = int(KSVD_QUERY_SAMPLE)   # row queried; defaults to the row from cell 8.16
FAISS_ATOM_EPS        = float(KSVD_QUERY_ACTIVE_EPS)
FAISS_ATOM_TOPK       = None    # None = threshold by FAISS_ATOM_EPS; int = keep top-K per row
FAISS_ATOM_NEIGHBORS  = 9       # neighbors shown in the thumbnail grid
FAISS_ATOM_THUMB_PX   = 220


def _threshold_codes_atoms(codes: np.ndarray, eps: float, topk: int | None) -> np.ndarray:
    out = codes.copy().astype(np.float32)
    if topk is None:
        out[np.abs(out) <= eps] = 0.0
        return out
    keep_k = max(1, min(int(topk), out.shape[1]))
    for i in range(out.shape[0]):
        order = np.argsort(np.abs(out[i]))[::-1][:keep_k]
        order = order[np.abs(out[i, order]) > eps]
        keep_mask = np.zeros(out.shape[1], dtype=bool)
        keep_mask[order] = True
        out[i, ~keep_mask] = 0.0
    return out


_thr_codes = _threshold_codes_atoms(C_all, FAISS_ATOM_EPS, FAISS_ATOM_TOPK)
_row_norms = np.linalg.norm(_thr_codes, axis=1)
_safe_norms = np.where(_row_norms > 0, _row_norms, 1.0)
_xb_atoms = (_thr_codes / _safe_norms[:, None]).astype(np.float32)
_xb_atoms[_row_norms == 0] = 0.0   # all-zero rows can't match anything; force IP=0
_xb_atoms = np.ascontiguousarray(_xb_atoms)

q_pattern = _xb_atoms[FAISS_ATOM_QUERY:FAISS_ATOM_QUERY + 1]
if float(np.linalg.norm(q_pattern)) == 0.0:
    raise ValueError(
        f"row {FAISS_ATOM_QUERY} has no active atoms with FAISS_ATOM_EPS={FAISS_ATOM_EPS}; "
        "lower FAISS_ATOM_EPS or set FAISS_ATOM_TOPK."
    )

atom_index = faiss.IndexFlatIP(_xb_atoms.shape[1])
atom_index.add(_xb_atoms)
_search_k = min(FAISS_ATOM_NEIGHBORS + 2, _xb_atoms.shape[0])
sim_scores, sim_idx = atom_index.search(q_pattern, _search_k)

_target_active = set(int(a) for a in np.flatnonzero(_thr_codes[FAISS_ATOM_QUERY] != 0))

def _shared_active(j: int) -> list[int]:
    js = set(int(a) for a in np.flatnonzero(_thr_codes[j] != 0))
    return sorted(_target_active & js, key=lambda a: -abs(float(_thr_codes[j, a])))


def _safe_meta(j: int, col: str) -> str:
    if col not in _df_manifest.columns:
        return ""
    v = _df_manifest.iloc[int(j)][col]
    return "" if (isinstance(v, float) and np.isnan(v)) else str(v).strip()


_neighbor_rows = []
_seen = set()
for ix, sc in zip(sim_idx[0].tolist(), sim_scores[0].tolist()):
    if ix < 0 or ix == FAISS_ATOM_QUERY or ix in _seen:
        continue
    _seen.add(ix)
    shared = _shared_active(int(ix))
    shared_top = shared[:8]
    shared_top_str = ", ".join(f"{a}:{float(_thr_codes[ix, a]):+.2f}" for a in shared_top)
    _neighbor_rows.append({
        "rank":               len(_neighbor_rows) + 1,
        "row":                int(ix),
        "image_idx":          int(MANIFEST_KEYS[int(ix)]),
        "file":               str(filenames[int(ix)]) if int(ix) < len(filenames) else f"row_{ix}",
        "cosine_sim":         float(sc),
        "n_active_atoms":     int((np.abs(_thr_codes[int(ix)]) > 0).sum()),
        "n_shared_atoms":     int(len(shared)),
        "top_shared_atoms":   shared_top_str,
        "illum":              _safe_meta(int(ix), "Illumination Modality"),
        "class":              _safe_meta(int(ix), "Class"),
        "subclass":           _safe_meta(int(ix), "Subclass"),
        "description":        _safe_meta(int(ix), "Description"),
        "tree_class_key":     _safe_meta(int(ix), "tree_class_key"),
    })
    if len(_neighbor_rows) >= FAISS_ATOM_NEIGHBORS:
        break

faiss_atom_neighbors_df = pd.DataFrame(_neighbor_rows)
print(
    f"FAISS atom-pattern query | row={FAISS_ATOM_QUERY} "
    f"| eps={FAISS_ATOM_EPS} | topk={FAISS_ATOM_TOPK} "
    f"| query_active_atoms={len(_target_active)} | neighbors_shown={len(faiss_atom_neighbors_df)}"
)
display(faiss_atom_neighbors_df)

# ---- Thumbnail grid: query first, then top neighbors ----
from PIL import Image as _PIL_Image

_grid_rows = [int(FAISS_ATOM_QUERY)] + [r["row"] for r in _neighbor_rows]
_n_cols = 5
_n_rows_grid = int(np.ceil(len(_grid_rows) / _n_cols))
fig_atom, axes_atom = plt.subplots(_n_rows_grid, _n_cols, figsize=(3.6 * _n_cols, 3.8 * _n_rows_grid))
axes_atom = np.atleast_1d(axes_atom).ravel()

for j, ridx in enumerate(_grid_rows):
    ax = axes_atom[j]
    img_name = str(filenames[ridx]) if ridx < len(filenames) else ""
    p = (IMAGES_DIR / img_name) if img_name else None
    if p and Path(p).is_file():
        try:
            with _PIL_Image.open(p) as im:
                im = im.convert("RGB")
                im.thumbnail((FAISS_ATOM_THUMB_PX, FAISS_ATOM_THUMB_PX))
                ax.imshow(im)
        except Exception:
            ax.text(0.5, 0.5, "image load error", ha="center", va="center")
    else:
        ax.text(0.5, 0.5, "missing image", ha="center", va="center")

    desc = _safe_meta(ridx, "Description")
    desc_head = (desc[:48] + "…") if len(desc) > 48 else desc
    if not desc_head:
        desc_head = "(no description)"
    if j == 0:
        ax.set_title(f"QUERY row {ridx}\n{desc_head}", fontsize=9)
    else:
        meta = _neighbor_rows[j - 1]
        ax.set_title(
            f"#{meta['rank']} row {ridx} | cos={meta['cosine_sim']:+.3f}\n"
            f"shared={meta['n_shared_atoms']} | {desc_head}",
            fontsize=9,
        )
    ax.set_xlabel(f"illum: {_safe_meta(ridx, 'Illumination Modality')[:40]}", fontsize=8)
    ax.axis("off")

for j in range(len(_grid_rows), len(axes_atom)):
    axes_atom[j].axis("off")

fig_atom.suptitle(
    f"FAISS neighbors in atom-strength space | query row {FAISS_ATOM_QUERY}",
    fontsize=12,
)
plt.tight_layout()
plt.show()

faiss_atom_neighbors_df.to_csv(
    KSVD_OUT_DIR / f"faiss_atom_neighbors_row{FAISS_ATOM_QUERY}.csv", index=False
)
print(f"-> wrote {KSVD_OUT_DIR / f'faiss_atom_neighbors_row{FAISS_ATOM_QUERY}.csv'}")

## 8.19 — What do *this image's* atoms actually express?

8.16 listed the active atoms for the chosen specimen, and 8.18 showed which other specimens
share the same atom pattern. This last cell zooms into the *atoms themselves*: for each
top-`SHOW_TOP_K` active atom in the query specimen, it shows

* a thumbnail panel of the **top exemplar images** for that atom (rows where the atom fires
  hardest in the same sign as the query — these are the visual concept the atom encodes), and
* a per-atom posterior table over `(Illumination Modality | Class | Subclass)` and the strongest
  classifier-decoded class names from `atom_logit_delta`, so you can read each atom as a
  small, named, retrievable concept.

Together with 8.16 + 8.18 this gives the loop the user asked for: pick an image → see its
atoms → use FAISS over atom strengths to find specimens that share those atoms → and for each
shared atom, surface what concept the atom actually expresses across the whole corpus.

In [ ]:
# 8.20 — For each top atom in the query specimen, render the atom's exemplar images +
# a compact posterior over (Illumination Modality / Class / Subclass) and the classifier head.
SHOW_TOP_K_ATOMS    = 6      # number of query-atoms to expand into image grids
EXEMPLARS_PER_ATOM  = 9
EXEMPLAR_THUMB_PX   = 200
COHORT_TOP_K        = 50     # rows used to estimate per-atom metadata posterior
COHORT_POST_TOP_N   = 5
LOGIT_TOP_N         = 5

if not _decode_rows:
    raise RuntimeError("Run cell 8.16 first; per_atom_decode_df is empty.")

_top_atoms = [int(r["atom"]) for r in _decode_rows[:SHOW_TOP_K_ATOMS]]
print(
    f"Query row {KSVD_QUERY_SAMPLE} | expanding top-{len(_top_atoms)} atoms "
    f"with {EXEMPLARS_PER_ATOM} exemplars each."
)


def _atom_top_exemplars(atom_id: int, *, top_k: int) -> tuple[np.ndarray, np.ndarray]:
    col = C_all[:, int(atom_id)].astype(np.float32)
    sign = 1.0 if sample_code[int(atom_id)] >= 0 else -1.0
    scores = sign * col
    order = np.argsort(scores)[::-1][: max(int(top_k), 1)]
    return order.astype(np.int64), scores[order].astype(np.float32)


def _posterior_over(col_name: str, rows: np.ndarray, weights: np.ndarray, top_n: int) -> pd.DataFrame:
    if col_name not in _df_manifest.columns:
        return pd.DataFrame(columns=[col_name, "posterior", "support"])
    vals = _df_manifest.iloc[rows][col_name].fillna("Unknown").astype(str).str.strip().to_numpy()
    w = weights.copy()
    w = np.maximum(w, 0.0)
    if w.sum() <= 0:
        w = np.ones_like(w)
    w = w / w.sum()
    s = pd.Series(w).groupby(vals).sum()
    sup = pd.Series(np.ones_like(w)).groupby(vals).sum()
    out = pd.DataFrame({col_name: s.index, "posterior": s.to_numpy(), "support": sup.reindex(s.index).to_numpy()})
    return out.sort_values("posterior", ascending=False).head(top_n).reset_index(drop=True)


def _atom_top_classes(atom_id: int, *, top_n: int, descending: bool) -> pd.DataFrame:
    sign = 1.0 if sample_code[int(atom_id)] >= 0 else -1.0
    deltas = sign * atom_logit_delta[int(atom_id)]
    if descending:
        ix = np.argsort(deltas)[::-1][:top_n]
    else:
        ix = np.argsort(deltas)[:top_n]
    return pd.DataFrame({
        "class_id":   [int(i) for i in ix],
        "class_name": [KSVD_CLASS_NAMES[int(i)] for i in ix],
        "logit_delta": [float(deltas[int(i)]) for i in ix],
    })


for _atom in _top_atoms:
    _w_query = float(sample_code[int(_atom)])
    _ex_rows, _ex_scores = _atom_top_exemplars(int(_atom), top_k=COHORT_TOP_K)
    _show_rows   = _ex_rows[:EXEMPLARS_PER_ATOM]
    _show_scores = _ex_scores[:EXEMPLARS_PER_ATOM]

    print("\n" + "=" * 78)
    print(
        f"Atom {int(_atom)} | query weight = {_w_query:+.3f} "
        f"| direction = {'positive' if _w_query >= 0 else 'negative'}"
    )
    print(f"  cohort size used for posterior = {len(_ex_rows)} rows")

    illum_post = _posterior_over("Illumination Modality", _ex_rows, _ex_scores, COHORT_POST_TOP_N)
    class_post = _posterior_over("Class",                  _ex_rows, _ex_scores, COHORT_POST_TOP_N)
    sub_post   = _posterior_over("Subclass",               _ex_rows, _ex_scores, COHORT_POST_TOP_N)
    desc_post  = _posterior_over("Description",            _ex_rows, _ex_scores, COHORT_POST_TOP_N)

    print("\n  posterior over Illumination Modality:")
    display(illum_post)
    print("  posterior over Class:")
    display(class_post)
    print("  posterior over Subclass:")
    display(sub_post)
    print("  posterior over Description:")
    display(desc_post)

    print("  classifier classes this atom pushes toward:")
    display(_atom_top_classes(int(_atom), top_n=LOGIT_TOP_N, descending=True))
    print("  classifier classes this atom pushes away from:")
    display(_atom_top_classes(int(_atom), top_n=LOGIT_TOP_N, descending=False))

    fig_e, axes_e = plt.subplots(3, 3, figsize=(3.4 * 3, 3.6 * 3))
    axes_e = np.atleast_1d(axes_e).ravel()
    for j, (_r, _s) in enumerate(zip(_show_rows, _show_scores)):
        ax = axes_e[j]
        img_name = str(filenames[int(_r)]) if int(_r) < len(filenames) else ""
        p = (IMAGES_DIR / img_name) if img_name else None
        if p and Path(p).is_file():
            try:
                with _PIL_Image.open(p) as im:
                    im = im.convert("RGB")
                    im.thumbnail((EXEMPLAR_THUMB_PX, EXEMPLAR_THUMB_PX))
                    ax.imshow(im)
            except Exception:
                ax.text(0.5, 0.5, "image load error", ha="center", va="center")
        else:
            ax.text(0.5, 0.5, "missing image", ha="center", va="center")
        _illum = _safe_meta(int(_r), "Illumination Modality")
        _desc  = _safe_meta(int(_r), "Description")
        _desc_short = (_desc[:36] + "…") if len(_desc) > 36 else _desc
        ax.set_title(
            f"#{j+1} row {int(_r)} | s={float(_s):+.2f}\n{_desc_short}",
            fontsize=8,
        )
        ax.set_xlabel(f"illum: {_illum[:32]}", fontsize=7)
        ax.axis("off")
    for j in range(len(_show_rows), len(axes_e)):
        axes_e[j].axis("off")
    fig_e.suptitle(
        f"Atom {int(_atom)} top exemplars (signed by query weight {_w_query:+.3f})",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

print("\nDone. Tweak SHOW_TOP_K_ATOMS / EXEMPLARS_PER_ATOM / COHORT_TOP_K to control the depth.")